In [10]:
from pathlib import Path
import json

import numpy as np
import pandas as pd


RESULTS_FOLDER = Path(
    "/Users/vladimir.kondratyev/minimal_volume_conformal_prediction/benchmark/results/scm20d"
)

In [11]:
records = []

for config_path in sorted(RESULTS_FOLDER.glob("*/seed_*/config.json")):
    config = json.loads(config_path.read_text())
    metrics = json.loads(config_path.with_name("metrics.json").read_text())

    model_type = config["predictor_config"]["type"]
    if config["rearrangement_config"] is not None:
        model_type += "_rearranged"

    calibrator = config["conformal_config"]["calibrator"]
    conformal_score_type = calibrator["type"]
    if conformal_score_type == "norm":
        conformal_score_type = f"l{calibrator['p']:g}"

    for metric_name, metric in metrics.items():
        if isinstance(metric, dict):
            records.append(
                {
                    "model_type": model_type,
                    "conformal_score_type": conformal_score_type,
                    "seed": config["seed"],
                    "metric": metric_name,
                    "seed_mean": metric["mean"],
                }
            )

results = pd.DataFrame(records)
results = results.loc[
    (results["metric"] != "log_volume_per_dimension")
    | np.isfinite(results["seed_mean"])
]
statistics = (
    results.groupby(["model_type", "conformal_score_type", "metric"])["seed_mean"]
    .agg(mean="mean", std="std")
)

In [12]:
for (model_type, conformal_score_type), table in statistics.groupby(
    level=["model_type", "conformal_score_type"]
):
    print("=" * 80)
    print(f"Model: {model_type}")
    print(f"Conformal score: {conformal_score_type}\n")
    print(
        table.droplevel(["model_type", "conformal_score_type"])
        .rename(
            columns={
                "mean": "Mean across seeds",
                "std": "Std of seed means",
            }
        )
        .to_string(float_format=lambda value: f"{value:.6f}")
    )
    print()

Model: neural_optimal_transport
Conformal score: l2

                          Mean across seeds  Std of seed means
metric                                                        
excess_coverage_risk               0.082284           0.005394
log_volume_per_dimension           5.820586           0.049506
marginal_coverage                  0.902007           0.009754
worst_slab_coverage                0.890659           0.038624

Model: neural_optimal_transport_rearranged
Conformal score: l2

                          Mean across seeds  Std of seed means
metric                                                        
excess_coverage_risk               0.082888           0.002271
log_volume_per_dimension           5.806680           0.044253
marginal_coverage                  0.903010           0.007952
worst_slab_coverage                0.882957           0.032168

Model: normalizing_flow
Conformal score: cdf_calibrator

                          Mean across seeds  Std of seed means
metri

In [15]:
# Add or remove result roots and metric names here.
RESULTS_FOLDERS = [
    Path(
        "/Users/vladimir.kondratyev/minimal_volume_conformal_prediction/"
        "benchmark/results/bio"
    ),
    Path(
        "/Users/vladimir.kondratyev/minimal_volume_conformal_prediction/"
        "benchmark/results/blog"
    ),
    Path(
        "/Users/vladimir.kondratyev/minimal_volume_conformal_prediction/"
        "benchmark/results/sgemm"
    ),
    Path(
        "/Users/vladimir.kondratyev/minimal_volume_conformal_prediction/"
        "benchmark/results/qm9"
    ),
    Path(
        "/Users/vladimir.kondratyev/minimal_volume_conformal_prediction/"
        "benchmark/results/scm20d"
    ),
]
SELECTED_METRICS = [
    "marginal_coverage",
]
# Case-insensitive substrings matched~ against experiment and model names.
# Set to None to include every model.
# SELECTED_MODELS = [
#     "rearranged",
#     "transport_realnvp_l2", "transport_neural_ot", 'transport_realnvp_cdf', 'transport_realnvp_log_probability']
SELECTED_MODELS = []

def print_selected_metrics(
    results_folders, selected_metrics, selected_models=None
):
    """Print selected metric means and cross-seed standard deviations."""
    summaries = {}
    model_patterns = tuple(
        str(model).casefold() for model in (selected_models or ())
    )

    for results_folder in map(Path, results_folders):
        print("=" * 120)
        print(results_folder.name)

        if not results_folder.is_dir():
            print("Folder does not exist.\n")
            continue

        config_paths = sorted(
            results_folder.glob("*/seed_*/config.json")
        )
        if not config_paths:
            print("No */seed_*/config.json files found.\n")
            continue

        records = []
        skipped_metrics_files = []
        for config_path in config_paths:
            metrics_path = config_path.with_name("metrics.json")
            if not metrics_path.is_file():
                skipped_metrics_files.append(metrics_path)
                continue

            config = json.loads(config_path.read_text(encoding="utf-8"))
            metrics = json.loads(metrics_path.read_text(encoding="utf-8"))

            model_type = config.get("predictor_config", {}).get(
                "type", "unknown"
            )
        
            if config.get("rearrangement_config") is not None:
                model_type += "_rearranged"

            conformal_config = config.get("conformal_config") or {}
            calibrator = conformal_config.get("calibrator") or {}
            score_type = calibrator.get("type", "unknown")
            if score_type == "norm":
                score_type = f"l{calibrator['p']:g}"

            experiment_name = config_path.parent.parent.name
            model_names = (experiment_name.casefold(), model_type.casefold())
            if model_patterns and not any(
                pattern in name
                for pattern in model_patterns
                for name in model_names
            ):
                continue

            record = {
                # "experiment": experiment_name,
                "model_type": model_type,
                "model_name": model_names[0]
                # "conformal_score_type": score_type,
                # "seed": config.get("seed", config_path.parent.name),
            }

            for metric_name in selected_metrics:
                metric = metrics.get(metric_name)
                if isinstance(metric, dict):
                    metric = metric.get("mean")
                record[metric_name] = (
                    float(metric)
                    if isinstance(metric, (int, float))
                    else np.nan
                )
            records.append(record)

        if skipped_metrics_files:
            print(
                f"Skipped {len(skipped_metrics_files)} runs without "
                "metrics.json."
            )
        if not records:
            print("No readable result records found.\n")
            continue

        seed_metrics = pd.DataFrame(records)

        seed_metrics[list(selected_metrics)] = seed_metrics[
            list(selected_metrics)
        ].replace([np.inf, -np.inf], np.nan)
        group_columns = [
            # "experiment",
            "model_name"
            # "conformal_score_type",
        ]
        grouped = seed_metrics.groupby(group_columns, sort=True)
        metric_statistics = grouped[list(selected_metrics)].agg(
            ["mean", "std"]
        )
        metric_statistics.columns = [
            f"{metric} ({statistic})"
            for metric, statistic in metric_statistics.columns
        ]
        summary = pd.concat(
            [
                # grouped["seed"].nunique().rename("n_seeds"),
                metric_statistics,
            ],
            axis=1,
        ).reset_index()

        summaries[str(results_folder)] = summary
        print(
            summary.to_string(
                index=False,
                na_rep="--",
                float_format=lambda value: f"{value:.6f}",
            )
        )
        print()

    return summaries


selected_metric_summaries = print_selected_metrics(
    RESULTS_FOLDERS, SELECTED_METRICS, SELECTED_MODELS
)

bio
                       model_name  marginal_coverage (mean)  marginal_coverage (std)
             residual_rf_elliptic                  0.901728                 0.003914
          residual_rf_global_otcp                  0.900022                 0.002920
           residual_rf_local_otcp                  0.904461                 0.006676
           transport_neural_ot_l2                  0.902165                 0.003269
transport_neural_ot_rearranged_l2                  0.903105                 0.001807
            transport_realnvp_cdf                  0.908944                 0.003677
             transport_realnvp_l2                  0.899038                 0.002426
transport_realnvp_log_probability                  0.901793                 0.001422
  transport_realnvp_rearranged_l2                  0.902515                 0.003296

blog
                       model_name  marginal_coverage (mean)  marginal_coverage (std)
             residual_rf_elliptic                  0.89

In [ ]:
========================================================================================================================
bio
                       model_name  log_volume_per_dimension (mean)  log_volume_per_dimension (std)  worst_slab_coverage (mean)  worst_slab_coverage (std)
           transport_neural_ot_l2                         4.534345                        0.007023                    0.870583                   0.017369
transport_neural_ot_rearranged_l2                         4.503624                        0.008162                    0.867887                   0.019114
             transport_realnvp_l2                         4.545964                        0.097305                    0.883952                   0.009736
  transport_realnvp_rearranged_l2                         4.404925                        0.011192                    0.886503                   0.008072
transport_realnvp_log_probability                         4.402263                        0.013063                    0.787739                   0.019296
            transport_realnvp_cdf                         4.461205                        0.011585                    0.902843                   0.019900

========================================================================================================================
blog
                       model_name  log_volume_per_dimension (mean)  log_volume_per_dimension (std)  worst_slab_coverage (mean)  worst_slab_coverage (std)
           transport_neural_ot_l2                         2.872795                        0.036468                    0.767621                   0.020459
transport_neural_ot_rearranged_l2                         2.825498                        0.029716                    0.765589                   0.023916
             transport_realnvp_l2                         2.970758                        0.153772                    0.815105                   0.009577
  transport_realnvp_rearranged_l2                         2.552090                        0.079216                    0.820965                   0.027973
            transport_realnvp_cdf                         2.756069                        0.122160                    0.852533                   0.028480
transport_realnvp_log_probability                         2.459317                        0.050208                    0.751795                   0.006474

========================================================================================================================
sgemm
                       model_name  log_volume_per_dimension (mean)  log_volume_per_dimension (std)  worst_slab_coverage (mean)  worst_slab_coverage (std)
           transport_neural_ot_l2                        -2.238571                        0.027217                    0.802851                   0.021415
transport_neural_ot_rearranged_l2                        -2.247350                        0.028579                    0.788616                   0.010593
             transport_realnvp_l2                        -1.745526                        0.045938                    0.863709                   0.020616
            transport_realnvp_cdf                        -1.265083                        0.162738                    0.846109                   0.011546
transport_realnvp_log_probability                        -1.334942                        0.168587                    0.814363                   0.012782

========================================================================================================================
qm9
                       model_name  log_volume_per_dimension (mean)  log_volume_per_dimension (std)  worst_slab_coverage (mean)  worst_slab_coverage (std)
      transport_neural_ot_l2_huge                         0.242862                        0.010974                    0.852050                   0.013723
transport_neural_ot_rearranged_l2                         0.234434                        0.010132                    0.853770                   0.008615
             transport_realnvp_l2                        -0.049116                        0.018090                    0.874922                   0.013638
  transport_realnvp_rearranged_l2                        -0.050105                        0.018732                    0.876804                   0.013568
            transport_realnvp_cdf                         0.347070                        0.021594                    0.901928                   0.008498
transport_realnvp_log_probability                         0.206339                        0.020242                    0.798600                   0.014673

========================================================================================================================
scm20d
                       model_name  log_volume_per_dimension (mean)  log_volume_per_dimension (std)  worst_slab_coverage (mean)  worst_slab_coverage (std)
           transport_neural_ot_l2                         5.820586                        0.049506                    0.890659                   0.038624
transport_neural_ot_rearranged_l2                         5.806680                        0.044253                    0.882957                   0.032168
             transport_realnvp_l2                         6.746640                        0.163685                    0.882106                   0.041383
  transport_realnvp_rearranged_l2                         6.501066                        0.109820                    0.908238                   0.009410
            transport_realnvp_cdf                         6.918567                        0.012256                    1.000000                   0.000000
transport_realnvp_log_probability                         6.373841                        0.031819                    0.861542                   0.038881